## VIX Index Correlation V6
### Volume Acceleration Analysis

In [2]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import fisher_exact
from sklearn.metrics import roc_auc_score
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.request_api_v2 import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension
from Modules.fibonacci_retracement import FibonacciRetracement
from Modules.fibonacci_extension import FibonacciExtension

In [3]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
fred_url = "https://api.stlouisfed.org/fred/series/observations"
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()
# Fibonacciエクステンションをプロットするクラスのインスタンスを作成
fibonacci_extension_v2 = FibonacciExtension() 
# Fibonacciリトレースメントを計算するクラスのインスタンスを作成
fibonacci_retracement = FibonacciRetracement()

In [5]:
# Date range for fetching API s
start = '2010-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
frequency = "d"
aggregation_method = "avg"

In [4]:
def extract_csv(
    start: str,
    end: str,
    output_dir: str,
    file_name: str,
    csv_df: pd.DataFrame,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_name
    # 重複するファイルがある場合は削除
    if output_path.exists():
        output_path.unlink()
    # 保存
    df = csv_df.copy()
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]
    df.to_csv(output_path)
    return output_path

#### VIX時系列を取得

In [5]:
vix_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^VIX",
    market=None,
)
vix_df.tail()

取得件数: 4439


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4434,1322446,^VIX,None,2026-10-20,17.309999,19.129999,17.250000,18.420000,0,17.106,...,14.941097,37.287629,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4435,1332259,^VIX,None,2026-10-21,16.680000,17.580000,16.510000,17.250000,0,17.522,...,14.939329,40.258687,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4436,1332252,^VIX,None,2026-10-22,19.030001,20.309999,17.709999,18.290001,0,17.670,...,14.930694,43.767609,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4437,1332260,^VIX,None,2026-10-23,19.450001,20.660000,18.870001,19.180000,0,18.302,...,14.955775,46.422778,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4438,1340463,^VIX,None,2026-10-26,18.389999,19.840000,17.790001,19.660000,0,18.560,...,15.000243,47.139930,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### S&P500時系列を取得

In [6]:
sp500_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^GSPC",
    market=None,
)
sp500_df.tail()

取得件数: 4437


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4432,1322447,^GSPC,None,2026-10-20,7712.569824,7748.419922,7668.029785,7679.009766,5004770000,7717.717969,...,7667.092991,10.475540,False,NaN,7723.472031,NaN,NaN,NaN,NaN,False
4433,1322449,^GSPC,None,2026-10-21,7757.169922,7770.399902,7709.339844,7736.430176,4921480000,7710.167969,...,7678.297169,14.249125,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4434,1332253,^GSPC,None,2026-10-22,7663.430176,7702.939941,7637.359863,7694.620117,5972620000,7711.629980,...,7684.296682,16.289593,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4435,1332261,^GSPC,None,2026-10-23,7623.680176,7660.149902,7612.069824,7637.180176,5062330000,7690.002051,...,7682.095087,15.555366,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4436,1340449,^GSPC,None,2026-10-26,7676.770020,7702.129883,7639.700195,7639.700195,5838070000,7677.388086,...,7677.272749,14.983352,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### Nasdaq100時系列を取得

In [7]:
qqq_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^IXIC",
    market=None,
)
qqq_df.tail()

取得件数: 4260


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4255,1322448,^IXIC,None,2026-10-20,26424.820312,26601.710938,26179.730469,26332.919922,7660360000,26558.719922,...,26526.314282,46.008708,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4256,1322483,^IXIC,None,2026-10-21,26702.859375,26778.900391,26507.070312,26601.730469,7818990000,26472.737891,...,26575.880210,46.059933,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4257,1332254,^IXIC,None,2026-10-22,26008.859375,26359.410156,25918.089844,26308.410156,9012120000,26458.266016,...,26568.604422,43.797490,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4258,1332262,^IXIC,None,2026-10-23,25955.339844,26111.619141,25866.150391,25893.949219,6554790000,26306.459766,...,26497.683270,38.120038,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4259,1340464,^IXIC,None,2026-10-26,26169.380859,26306.820312,25986.560547,26026.869141,7402660000,26232.775781,...,26436.095556,33.159737,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### UVXY時系列を取得

In [8]:
uvxy_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="UVXY",
    market=None,
)
uvxy_df.tail()

取得件数: 3787


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
3782,1326266,UVXY,None,2026-10-20,17.180000,18.139999,16.809999,17.870001,17214800,16.916,...,16.8868,-89.780586,True,16.916,NaN,NaN,NaN,NaN,NaN,False
3783,1326267,UVXY,None,2026-10-21,16.799999,17.420000,16.709999,16.870001,14922300,17.090,...,16.8416,-88.875608,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3784,1332255,UVXY,None,2026-10-22,18.410000,18.969999,17.170000,17.459999,23934800,17.156,...,16.8448,-85.575856,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3785,1332263,UVXY,None,2026-10-23,18.760000,19.860001,18.610001,19.170000,20086400,17.778,...,16.9132,-78.233587,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3786,1340465,UVXY,None,2026-10-26,18.090000,18.370001,17.629999,18.270000,8059400,17.928,...,16.9600,-73.093998,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### SVXY時系列を取得

In [9]:
svxy_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="SVXY",
    market=None,
)
svxy_df.tail()

取得件数: 3787


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
3782,1330050,SVXY,None,2026-10-20,63.540001,63.990002,62.369999,62.689999,2225300,64.030,...,63.423886,87.483992,False,NaN,64.1664,NaN,NaN,NaN,NaN,False
3783,1330051,SVXY,None,2026-10-21,63.900002,64.150002,63.259998,63.930000,1611100,63.740,...,63.503574,86.096645,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3784,1332256,SVXY,None,2026-10-22,61.980000,63.520000,61.259998,63.250000,3109700,63.628,...,63.470040,82.784086,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3785,1332264,SVXY,None,2026-10-23,61.639999,61.740002,60.340000,61.150002,1988200,62.822,...,63.175975,74.280714,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3786,1340466,SVXY,None,2026-10-26,62.220001,62.810001,62.009998,62.160000,1230300,62.636,...,63.030542,67.425083,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### VXX時系列を取得

In [10]:
vxx_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="VXX",
    market=None,
)
vxx_df.tail()

取得件数: 2200


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
2195,1332247,VXX,None,2026-10-20,17.570000,18.280001,17.379999,18.08,13604000,17.421,...,16.985904,-88.474345,True,17.421,NaN,NaN,NaN,NaN,NaN,False
2196,1332248,VXX,None,2026-10-21,17.389999,17.785000,17.299999,17.40,10846000,17.545,...,16.987596,-87.526680,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2197,1332257,VXX,None,2026-10-22,18.480000,18.865000,17.639999,17.77,19858300,17.585,...,16.987295,-84.299496,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2198,1332265,VXX,None,2026-10-23,18.719999,19.450001,18.625000,18.99,16171200,18.021,...,16.917382,-75.898574,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2199,1340467,VXX,None,2026-10-26,18.309999,18.475000,17.980000,18.40,8749900,18.128,...,16.914875,-70.135747,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### データ統合

In [11]:
# データフレームのコピーを作成
vix_work = vix_df.copy()
sp500_work = sp500_df.copy()
qqq_work = qqq_df.copy()
uvxy_work = uvxy_df.copy()
svxy_work = svxy_df.copy()
vxx_work = vxx_df.copy()
# 必要列のみ抽出
vix_work = vix_work[["date", "close"]].rename(
    columns={"close": "vix"}
)
# S&P500の必要列のみ抽出
sp500_work = sp500_work[["date", "close"]].rename(
    columns={"close": "sp500"}
)
# QQQの必要列のみ抽出
qqq_work = qqq_work[["date", "close"]].rename(
    columns={"close": "qqq"}
)
# UVXYの必要列のみ抽出
uvxy_work = uvxy_work[["date", "close", "volume"]].rename(
    columns={
        "close": "uvxy",
        "volume": "uvxy_volume"
    }
)
# SVXYの必要列のみ抽出
svxy_work = svxy_work[["date", "close", "volume"]].rename(
    columns={
        "close": "svxy",
        "volume": "svxy_volume"
    }
)
# VXXの必要列のみ抽出
vxx_work = vxx_work[["date", "close", "volume"]].rename(
    columns={
        "close": "vxx",
        "volume": "vxx_volume"
    }
)
# 結合用のデータフレームを作成
research_df = (
    vix_work
    .merge(sp500_work, on="date",how="left")
    .merge(qqq_work, on="date", how="left")
    .merge(uvxy_work, on="date", how="left")
    .merge(svxy_work, on="date", how="left")
    .merge(vxx_work, on="date", how="left")
)
# 結合結果の形状を確認
print(research_df.shape)
# CSVファイルとして抽出
extract_csv(
    start=start,
    end=end,
    output_dir=output_dir,
    file_name=f"research_df_{start}_{end}.csv",
    csv_df=research_df,
)
# データフレームの先頭を表示
display(research_df.head())

(4439, 10)


,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume
0,2009-03-09,49.349998,680.760010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-03-10,49.680000,679.280029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-03-11,43.349998,719.590027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-03-12,43.529999,720.890015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-03-13,40.990002,751.969971,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 1. Volume ROC特徴量作成

##### 目的

出来高モメンタムを計算する。

- ROC5
- ROC10
- ROC20

In [12]:
# ボリューム列の変化率を計算
VOLUME_COLS = ["uvxy_volume", "svxy_volume", "vxx_volume"]
# 各ボリューム列の変化率を計算する日数のリスト
for volume_col in VOLUME_COLS:
    # 各日数に対して変化率を計算
    for n in [5, 10, 20]:
        # 変化率を計算して新しい列に格納
        research_df[f"{volume_col}_roc_{n}d"
        ] = (research_df[volume_col] - research_df[volume_col].shift(n))
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,uvxy_volume_roc_5d,uvxy_volume_roc_10d,uvxy_volume_roc_20d,svxy_volume_roc_5d,svxy_volume_roc_10d,svxy_volume_roc_20d,vxx_volume_roc_5d,vxx_volume_roc_10d,vxx_volume_roc_20d
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,12116600.0,10962100.0,8882600.0,1282500.0,977200.0,1322600.0,8772800.0,3593500.0,5568000.0
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,233900.0,7468100.0,5806500.0,-1015100.0,89400.0,501300.0,-2074300.0,2294800.0,2425000.0
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,10618500.0,14047400.0,14627200.0,1438000.0,1522900.0,2263400.0,7825400.0,9801700.0,10758900.0
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,-23883400.0,15690300.0,13630700.0,-11673800.0,1191000.0,966200.0,-28258700.0,9611200.0,9313700.0
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,-10820400.0,2739500.0,-394600.0,-2620000.0,417600.0,203300.0,-15466300.0,-881000.0,773700.0


#### 2. Volume Change特徴量作成

##### 目的

N営業日前からの出来高変化率を計算する。

- Change5
- Change10
- Change20

In [13]:
# ボリューム列の変化率を計算
for volume_col in VOLUME_COLS:
    # 各日数に対して変化率を計算
    for n in [5, 10, 20]:
        # 過去n日分の値を取得し、0をNaNに置換
        base = (
            research_df[volume_col]
            .shift(n)
            .replace(0, np.nan)
        )
        # 変化率を計算して新しい列に格納
        research_df[f"{volume_col}_change_{n}d"] = (research_df[volume_col] / base - 1)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_roc_20d,uvxy_volume_change_5d,uvxy_volume_change_10d,uvxy_volume_change_20d,svxy_volume_change_5d,svxy_volume_change_10d,svxy_volume_change_20d,vxx_volume_change_5d,vxx_volume_change_10d,vxx_volume_change_20d
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,5568000.0,2.376643,1.753179,1.066057,1.360310,0.782950,1.465160,1.815864,0.358973,0.692882
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,2425000.0,0.015924,1.001865,0.636971,-0.386528,0.058750,0.451703,-0.160546,0.268360,0.287971
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,10758900.0,0.797406,1.420738,1.571533,0.860202,0.959730,2.674465,0.650334,0.974653,1.182375
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,9313700.0,-0.543177,3.569141,2.111421,-0.854472,1.493979,0.945401,-0.636029,1.465122,1.358177
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,773700.0,-0.573120,0.514953,-0.046676,-0.680466,0.513843,0.197955,-0.638676,-0.091476,0.097001


#### 3. Volume Acceleration特徴量作成

##### 目的

出来高モメンタムの変化速度を計算する。

Acceleration A

ROC5 - ROC10

Acceleration B

ROC10 - ROC20

In [14]:
# ボリューム列の加速度を計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対して加速度を計算
    research_df[f"{volume_col}_accel_a"] = (
        research_df[f"{volume_col}_roc_5d"]
        -
        research_df[f"{volume_col}_roc_10d"]
    )
    # 各ボリューム列に対して加速度bを計算
    research_df[f"{volume_col}_accel_b"] = (
        research_df[f"{volume_col}_roc_10d"]
        -
        research_df[f"{volume_col}_roc_20d"]
    )
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,svxy_volume_change_20d,vxx_volume_change_5d,vxx_volume_change_10d,vxx_volume_change_20d,uvxy_volume_accel_a,uvxy_volume_accel_b,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,1.465160,1.815864,0.358973,0.692882,1154500.0,2079500.0,305300.0,-345400.0,5179300.0,-1974500.0
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,0.451703,-0.160546,0.268360,0.287971,-7234200.0,1661600.0,-1104500.0,-411900.0,-4369100.0,-130200.0
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,2.674465,0.650334,0.974653,1.182375,-3428900.0,-579800.0,-84900.0,-740500.0,-1976300.0,-957200.0
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,0.945401,-0.636029,1.465122,1.358177,-39573700.0,2059600.0,-12864800.0,224800.0,-37869900.0,297500.0
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,0.197955,-0.638676,-0.091476,0.097001,-13559900.0,3134100.0,-3037600.0,214300.0,-14585300.0,-1654700.0


#### 4. Volume Acceleration d(ROC)/dt特徴量作成

##### 目的

ROCの一次微分を計算する。

d(ROC)/dt

In [15]:
# ボリューム列のROC加速度を計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対してROC加速度を計算
    research_df[f"{volume_col}_roc_acceleration"] = (
        research_df[f"{volume_col}_roc_10d"]
        -
        research_df[f"{volume_col}_roc_10d"].shift(1)
    )
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_change_20d,uvxy_volume_accel_a,uvxy_volume_accel_b,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,0.692882,1154500.0,2079500.0,305300.0,-345400.0,5179300.0,-1974500.0,-3087000.0,-2191500.0,-13191700.0
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,0.287971,-7234200.0,1661600.0,-1104500.0,-411900.0,-4369100.0,-130200.0,-3494000.0,-887800.0,-1298700.0
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,1.182375,-3428900.0,-579800.0,-84900.0,-740500.0,-1976300.0,-957200.0,6579300.0,1433500.0,7506900.0
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,1.358177,-39573700.0,2059600.0,-12864800.0,224800.0,-37869900.0,297500.0,1642900.0,-331900.0,-190500.0
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,0.097001,-13559900.0,3134100.0,-3037600.0,214300.0,-14585300.0,-1654700.0,-12950800.0,-773400.0,-10492200.0


#### 5. Acceleration Z-Score作成

##### 目的

Acceleration異常度を標準化する。

In [16]:
# ボリューム列のROC加速度のZスコアを計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対してROC加速度のZスコアを計算
    accel_col = (f"{volume_col}_roc_acceleration")
    mean_col = (
        research_df[accel_col]
        .rolling(60)
        .mean()
    )
    std_col = (
        research_df[accel_col]
        .rolling(60)
        .std()
        .replace(0, np.nan)
    )
    research_df[f"{accel_col}_z"] = (research_df[accel_col] - mean_col) / std_col
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,305300.0,-345400.0,5179300.0,-1974500.0,-3087000.0,-2191500.0,-13191700.0,NaN,NaN,NaN
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,-1104500.0,-411900.0,-4369100.0,-130200.0,-3494000.0,-887800.0,-1298700.0,NaN,NaN,NaN
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,-84900.0,-740500.0,-1976300.0,-957200.0,6579300.0,1433500.0,7506900.0,NaN,NaN,NaN
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,-12864800.0,224800.0,-37869900.0,297500.0,1642900.0,-331900.0,-190500.0,NaN,NaN,NaN
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,-3037600.0,214300.0,-14585300.0,-1654700.0,-12950800.0,-773400.0,-10492200.0,NaN,NaN,NaN


#### 6. QQQ急落イベント作成

##### 目的

分類問題用ターゲットを作成する。

In [17]:
# QQQの1日下落を閾値ごとに計算
for threshold in [2, 3, 5]:
    research_df[f"qqq_1d_drop_{threshold}"] = (
        (research_df["qqq"].shift(-1) / research_df["qqq"] - 1) <= -(threshold / 100)
    ).astype(int)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,qqq_1d_drop_3,qqq_1d_drop_5
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,-1974500.0,-3087000.0,-2191500.0,-13191700.0,NaN,NaN,NaN,0,0,0
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,-130200.0,-3494000.0,-887800.0,-1298700.0,NaN,NaN,NaN,0,0,0
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,-957200.0,6579300.0,1433500.0,7506900.0,NaN,NaN,NaN,0,0,0
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,297500.0,1642900.0,-331900.0,-190500.0,NaN,NaN,NaN,0,0,0
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,-1654700.0,-12950800.0,-773400.0,-10492200.0,NaN,NaN,NaN,0,0,0


#### 7. VIX急騰イベント作成

##### 目的

VIX急騰ターゲットを作成する。

In [18]:
# VIXの1日リターンとスパイクを計算
research_df["vix_1d_return"] = (research_df["vix"].shift(-1) / research_df["vix"] - 1)
# VIXの1日リターンのスパイクを閾値ごとに計算
for threshold in [5, 10, 20]:
    research_df[f"vix_1d_spike_{threshold}"] = (
        research_df["vix_1d_return"] >= threshold / 100
    ).astype(int)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,qqq_1d_drop_3,qqq_1d_drop_5,vix_1d_return,vix_1d_spike_5,vix_1d_spike_10,vix_1d_spike_20
4434,2026-10-20,18.420000,7679.009766,26332.919922,17.870001,17214800.0,62.689999,2225300.0,18.08,13604000.0,...,NaN,NaN,NaN,0,0,0,-0.063518,0,0,0
4435,2026-10-21,17.250000,7736.430176,26601.730469,16.870001,14922300.0,63.930000,1611100.0,17.40,10846000.0,...,NaN,NaN,NaN,0,0,0,0.060290,1,0,0
4436,2026-10-22,18.290001,7694.620117,26308.410156,17.459999,23934800.0,63.250000,3109700.0,17.77,19858300.0,...,NaN,NaN,NaN,0,0,0,0.048660,0,0,0
4437,2026-10-23,19.180000,7637.180176,25893.949219,19.170000,20086400.0,61.150002,1988200.0,18.99,16171200.0,...,NaN,NaN,NaN,0,0,0,0.025026,0,0,0
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8059400.0,62.160000,1230300.0,18.40,8749900.0,...,NaN,NaN,NaN,0,0,0,NaN,0,0,0


#### 8. ROC-AUC分析

##### 目的

AccelerationとROCの説明力を比較する。

In [19]:
# 特徴量とターゲットの組み合わせに対してROC AUCを算出する
FEATURES = [c for c in research_df.columns
    if any(key in c for key in [
            "_roc_",
            "_accel_",
            "_change_"
        ]
    )
]
# ターゲット列を定義する
TARGETS = [
    "qqq_1d_drop_2",
    "qqq_1d_drop_3",
    "qqq_1d_drop_5",
    "vix_1d_spike_5",
    "vix_1d_spike_10",
    "vix_1d_spike_20"
]
rows = []
# ROC AUCを計算するためのループを回す
for feature in FEATURES:
    # 各ターゲットに対してROC AUCを計算
    for target in TARGETS:
        # 特徴量とターゲットの組み合わせに対してデータを抽出
        work_df = (
            research_df[[feature, target]]
            .replace([np.inf, -np.inf], np.nan).dropna()
        )
        # ターゲット列のユニークな値が2未満の場合はスキップ
        if (work_df[target].nunique() < 2):
            continue
        # ROC AUCを計算
        auc = roc_auc_score(
            work_df[target],
            work_df[feature]
        )
        # 計算結果をリストに追加
        rows.append({
            "feature": feature,
            "target": target,
            "roc_auc": auc
        })
# 計算結果をDataFrameに変換
auc_df = pd.DataFrame(rows)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"8_roc_auc_analysis_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = auc_df.copy()
df.to_csv(output_path)
display(
    auc_df
    .sort_values("roc_auc", ascending=False).head(100)
)

,feature,target,roc_auc
26,svxy_volume_roc_10d,qqq_1d_drop_5,0.950222
20,svxy_volume_roc_5d,qqq_1d_drop_5,0.895294
80,svxy_volume_change_10d,qqq_1d_drop_5,0.886098
32,svxy_volume_roc_20d,qqq_1d_drop_5,0.881799
86,svxy_volume_change_20d,qqq_1d_drop_5,0.867705
...,...,...,...
16,uvxy_volume_roc_20d,vix_1d_spike_10,0.645974
174,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,0.645244
4,uvxy_volume_roc_5d,vix_1d_spike_10,0.645088
148,uvxy_volume_roc_acceleration,vix_1d_spike_10,0.644333


#### 9. Top Feature抽出

##### 目的

Volume AccelerationがVolume ROCを上回るか確認する。

In [21]:
best_features_df = (
    auc_df
    .query("roc_auc >= 0.70")
    .sort_values("roc_auc", ascending=False)
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"9_roc_auc_top_features_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = best_features_df.copy()
df.to_csv(output_path)
display(
    best_features_df
)

,feature,target,roc_auc
26,svxy_volume_roc_10d,qqq_1d_drop_5,0.950222
20,svxy_volume_roc_5d,qqq_1d_drop_5,0.895294
80,svxy_volume_change_10d,qqq_1d_drop_5,0.886098
32,svxy_volume_roc_20d,qqq_1d_drop_5,0.881799
86,svxy_volume_change_20d,qqq_1d_drop_5,0.867705
74,svxy_volume_change_5d,qqq_1d_drop_5,0.859813
29,svxy_volume_roc_10d,vix_1d_spike_20,0.783563
170,svxy_volume_roc_acceleration_z,qqq_1d_drop_5,0.765091
83,svxy_volume_change_10d,vix_1d_spike_20,0.764271
152,svxy_volume_roc_acceleration,qqq_1d_drop_5,0.763803


# VIX Index Correlation V6 - Task 1.1 結論

## 検証目的

V5で有効性が確認された Volume ROC に対して、

- Volume Change
- Volume Acceleration
- Acceleration Z-Score

がより早期かつ強力な市場ストレスシグナルとなるか検証した。

対象ETFは

- SVXY
- VXX
- UVXY

とし、

- QQQ翌日急落
- VIX翌日急騰

に対する説明力をROC-AUCで評価した。 【1-0bfd35】【2-5f5688】

---

## 検証結果

最も高いROC-AUCを示した特徴量は引き続き

- SVXY Volume ROC 10D

であった。

QQQ翌日-5%に対して

```text
ROC-AUC = 0.950
```

を記録し、V5の結果を再現した。 【1-0bfd35】【2-5f5688】

また、

- SVXY Volume ROC 5D
- SVXY Volume Change 10D
- SVXY Volume ROC 20D

も高い説明力を示した。 【1-0bfd35】

---

## Volume Acceleration評価

本研究の主仮説であった

```text
Volume Acceleration > Volume ROC
```

は支持されなかった。

代表的な特徴量である

- SVXY Volume ROC Acceleration
- SVXY Volume ROC Acceleration Z

のROC-AUCは

```text
約0.76
```

に留まり、

SVXY Volume ROC 10D の

```text
0.95
```

を大きく下回った。 【1-0bfd35】【2-5f5688】

---

## 解釈

市場ストレス検知において重要なのは

```text
出来高増加速度
```

ではなく、

```text
出来高モメンタムの蓄積量
```

である可能性が高い。

つまり、

市場参加者のヘッジ需要やリスクオフ行動は

Accelerationではなく

ROCとして観測されていると考えられる。 【1-0bfd35】【2-5f5688】

---

## 結論

Task 1.1 の結果から、

Volume Acceleration単独によるLater1シグナルの改善は確認できなかった。

現時点の最有力特徴量は引き続き

```text
SVXY Volume ROC 10D
```

である。

したがって、

Volume Acceleration研究は一旦終了し、

次フェーズである

```text
Task 2.1
Price × Volume 統合分析
```

へ移行する。